# HW – Part 2: Classification with PyTorch → ONNX → GitHub Pages
**Dataset:** Pima Indians Diabetes (National Institute of Diabetes and Digestive and Kidney Diseases; UCI / Smith et al., 1988)

**Goal:** predict whether a patient (female, ≥ 21 years, Pima heritage) has diabetes (onset within 5 years) from 8 clinical measurements. Binary classification, class 1 = diabetes.

Pipeline: read data → EDA (class balance, correlation matrix, hidden missing values) → median imputation + standardization (stored **inside** the model) → stratified train/val/test split → 5 PyTorch classifiers (Logistic Regression, MLP, Deep NN, F1+F2, ResidualNet) → metrics (accuracy, precision, recall, **F-measure**, ROC-AUC, confusion matrix) → analysis → ONNX export for the web app.

In [ ]:
import os, json, time, copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report,
                             matthews_corrcoef, precision_recall_curve)

np.set_printoptions(suppress=True)
torch.set_printoptions(sci_mode=False)

OUT_FIG = "outputs/figures"
OUT_RES = "outputs/results"
OUT_WEB = "outputs/docs/classification"
for d in [OUT_FIG, OUT_RES, OUT_WEB]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 1. Read the data

In [ ]:
COLS = ["pregnancies", "glucose", "blood_pressure", "skin_thickness", "insulin",
        "bmi", "dpf", "age", "outcome"]
FEATURES = COLS[:-1]
TARGET = "outcome"
UNITS = {"pregnancies": "count", "glucose": "mg/dL (2-h OGTT)", "blood_pressure": "mm Hg (diastolic)",
         "skin_thickness": "mm (triceps)", "insulin": "µU/mL (2-h serum)", "bmi": "kg/m²",
         "dpf": "diabetes pedigree function", "age": "years"}
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"
LOCAL = "pima.csv"
df_raw = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL, header=None, names=COLS)
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.describe().T

## 2. Exploratory data analysis

In [ ]:
print("Missing (NaN):", int(df_raw.isna().sum().sum()), "| duplicates:", int(df_raw.duplicated().sum()))
counts = df_raw[TARGET].value_counts().sort_index()
print("Class balance:", counts.to_dict(), "→ positive rate =", round(counts[1] / counts.sum(), 3))

ZERO_AS_MISSING = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
zeros = (df_raw[ZERO_AS_MISSING] == 0).sum()
print("\nPhysiologically impossible zeros (= hidden missing values):")
print(pd.DataFrame({"zeros": zeros, "percent": (zeros / len(df_raw) * 100).round(1)}))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.ravel(), FEATURES):
    sns.histplot(data=df_raw, x=col, hue=TARGET, bins=30, ax=ax, element="step",
                 palette={0: "#3b6ea5", 1: "#c0504d"})
    ax.set_title(col)
sns.countplot(x=TARGET, data=df_raw, ax=axes[2, 2], hue=TARGET, palette={0: "#3b6ea5", 1: "#c0504d"}, legend=False)
axes[2, 2].set_title("class balance (0 = no diabetes, 1 = diabetes)")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/clf_distributions.png", dpi=150); plt.show()

**Observations**
* No NaNs, but zeros in glucose, blood pressure, skin thickness, insulin and BMI are physiologically impossible → they are **missing values encoded as 0** (insulin ≈ 49 %, skin thickness ≈ 30 %).
* Classes are imbalanced (≈ 65 % / 35 %) → accuracy alone is misleading; use **F-measure**, recall, ROC-AUC, and a class-weighted loss.

## 3. Processing and split (stratified 70 / 15 / 15)
* Zeros → NaN → **median of the training set** (computed only on non-zero training values, so no leakage).
* Standardization statistics also come from the training set only.
* Both steps are stored as buffers inside the model so the ONNX file does the exact same processing in the browser.

In [ ]:
X = df_raw[FEATURES].values.astype(np.float32)
y = df_raw[TARGET].values.astype(np.int64)

SPLIT_SEED = 42
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SPLIT_SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SPLIT_SEED, stratify=y_tmp)
print("train", X_train.shape, np.bincount(y_train), "| val", X_val.shape, np.bincount(y_val),
      "| test", X_test.shape, np.bincount(y_test))

zero_mask = torch.tensor([f in ZERO_AS_MISSING for f in FEATURES])
medians = torch.tensor([np.median(X_train[X_train[:, j] > 0, j]) if FEATURES[j] in ZERO_AS_MISSING
                        else np.median(X_train[:, j]) for j in range(len(FEATURES))], dtype=torch.float32)
print("Training medians used for imputation:", dict(zip(FEATURES, medians.numpy().round(3))))

def impute(x, zero_mask, medians):
    return torch.where((x == 0) & zero_mask, medians.expand_as(x), x)

Xtr_imp = impute(torch.tensor(X_train), zero_mask, medians)
x_means, x_deviations = Xtr_imp.mean(0), Xtr_imp.std(0)

# Correlation matrix AFTER imputation (training set)
df_imp = pd.DataFrame(Xtr_imp.numpy(), columns=FEATURES); df_imp[TARGET] = y_train
plt.figure(figsize=(9, 7))
corr = df_imp.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
plt.title("Pima – feature correlation matrix (train, after imputation)")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/clf_correlation.png", dpi=150); plt.show()
print(corr[TARGET].drop(TARGET).sort_values(ascending=False))

# class weights for the imbalanced loss
cw = torch.tensor(len(y_train) / (2 * np.bincount(y_train)), dtype=torch.float32)
print("class weights:", cw.numpy())

to_t = lambda a: torch.tensor(a, dtype=torch.float32)
train_ds = TensorDataset(to_t(X_train), torch.tensor(y_train))

## 4. Models (PyTorch, as in class)
Every model: impute → standardize → network → **2 logits**. Training uses `CrossEntropyLoss` (softmax inside); the exported ONNX model outputs **softmax probabilities** `[P(no diabetes), P(diabetes)]`.

| Model | Idea |
|---|---|
| **Logistic Regression** | `nn.Linear(8,2)` – linear decision boundary |
| **MLP** | one hidden layer (32, ReLU) |
| **Deep NN** | 64-32-16 ReLU with dropout |
| **F1+F2** | linear path + non-linear sigmoid path (class model) |
| **ResidualNet** | 2×64 ReLU + skip connection (class model) |

In [ ]:
class ScaledClassifier(nn.Module):
    def __init__(self, zero_mask, medians, x_means, x_deviations):
        super().__init__()
        self.register_buffer("zero_mask", zero_mask.clone())
        self.register_buffer("medians", medians.clone())
        self.register_buffer("x_means", x_means.clone())
        self.register_buffer("x_deviations", x_deviations.clone())

    def core(self, x):
        raise NotImplementedError

    def forward(self, x):                      # returns logits
        x = impute(x, self.zero_mask, self.medians)
        x = (x - self.x_means) / self.x_deviations
        return self.core(x)


class LogisticReg(ScaledClassifier):
    def __init__(self, *s, n_in=8):
        super().__init__(*s); self.linear1 = nn.Linear(n_in, 2)
    def core(self, x):
        return self.linear1(x)


class MLP(ScaledClassifier):
    def __init__(self, *s, n_in=8, hidden=32):
        super().__init__(*s)
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, 2))
    def core(self, x):
        return self.net(x)


class DeepNN(ScaledClassifier):
    def __init__(self, *s, n_in=8, p=0.2):
        super().__init__(*s)
        self.net = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(32, 16), nn.ReLU(),
                                 nn.Linear(16, 2))
    def core(self, x):
        return self.net(x)


class F1plusF2(ScaledClassifier):
    def __init__(self, *s, n_in=8, hidden=10):
        super().__init__(*s)
        self.f1_linear1 = nn.Linear(n_in, 2)
        self.f2_linear1 = nn.Linear(n_in, hidden); self.f2_act1 = nn.Sigmoid()
        self.f2_dropout = nn.Dropout(0.25);        self.f2_linear2 = nn.Linear(hidden, 2)
    def core(self, x):
        return self.f1_linear1(x) + self.f2_linear2(self.f2_dropout(self.f2_act1(self.f2_linear1(x))))


class ResidualNet(ScaledClassifier):
    def __init__(self, *s, n_in=8, hidden=64, dropout_rate=0.2):
        super().__init__(*s)
        self.fc1 = nn.Linear(n_in, hidden);   self.act1 = nn.ReLU(); self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden, hidden); self.act2 = nn.ReLU(); self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(hidden, 2);      self.input_proj = nn.Linear(n_in, hidden)
    def core(self, x):
        x0 = self.input_proj(x)
        x1 = self.dropout1(self.act1(self.fc1(x)))
        x2 = self.dropout2(self.act2(self.fc2(x1)))
        return self.fc3(x2 + x0)


MODELS = {"Logistic Regression": LogisticReg, "MLP": MLP, "Deep NN": DeepNN,
          "F1+F2": F1plusF2, "ResidualNet": ResidualNet}

## 5. Training loop
* Loss: class-weighted `CrossEntropyLoss` · Adam (lr = 1e-3, weight decay 1e-4) · batch = 32
* Up to 1000 epochs, early stopping on validation loss (patience 60), best weights restored.
* 5 seeds per architecture → mean ± std.

In [ ]:
EPOCHS, PATIENCE, BATCH, LR = 1000, 60, 32, 1e-3
SEEDS = [0, 1, 2, 3, 4]
STATS = (zero_mask, medians, x_means, x_deviations)

def predict_proba(model, X_np):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(to_t(X_np).to(device)), dim=1)[:, 1].cpu().numpy()

def train_model(cls, seed):
    set_seed(seed)
    model = cls(*STATS).to(device)
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(weight=cw.to(device))
    dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, generator=torch.Generator().manual_seed(seed))
    Xv, yv = to_t(X_val).to(device), torch.tensor(y_val).to(device)
    best, best_state, wait, hist = float("inf"), None, 0, {"train": [], "val": []}
    for epoch in range(EPOCHS):
        model.train(); run = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step()
            run += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xv), yv).item()
        hist["train"].append(run / len(train_ds)); hist["val"].append(vl)
        if vl < best - 1e-6:
            best, best_state, wait = vl, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, hist

def clf_metrics(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {"accuracy": float(accuracy_score(y_true, pred)),
            "precision": float(precision_score(y_true, pred, zero_division=0)),
            "recall": float(recall_score(y_true, pred)),
            "F1": float(f1_score(y_true, pred)),
            "F1_macro": float(f1_score(y_true, pred, average="macro")),
            "MCC": float(matthews_corrcoef(y_true, pred)),
            "ROC_AUC": float(roc_auc_score(y_true, proba))}

In [ ]:
rows, trained, histories = [], {}, {}
t0 = time.time()
for name, cls in MODELS.items():
    for seed in SEEDS:
        model, hist = train_model(cls, seed)
        mv = clf_metrics(y_val, predict_proba(model, X_val))
        mt = clf_metrics(y_test, predict_proba(model, X_test))
        rows.append({"model": name, "seed": seed, "epochs": len(hist["val"]),
                     "val_F1": mv["F1"], "val_AUC": mv["ROC_AUC"],
                     **{f"test_{k}": v for k, v in mt.items()}})
        trained[(name, seed)] = model; histories[(name, seed)] = hist
    r = pd.DataFrame([x for x in rows if x["model"] == name])
    print(f"{name:20s} test F1 = {r.test_F1.mean():.3f} ± {r.test_F1.std():.3f} | "
          f"AUC = {r.test_ROC_AUC.mean():.3f}   ({time.time()-t0:.0f}s)")

runs = pd.DataFrame(rows)
runs.to_csv(f"{OUT_RES}/classification_all_runs.csv", index=False)

## 6. Results

In [ ]:
metric_cols = ["accuracy", "precision", "recall", "F1", "F1_macro", "MCC", "ROC_AUC"]
summary = runs.groupby("model").agg(
    val_F1=("val_F1", "mean"),
    **{f"{m}": (f"test_{m}", "mean") for m in metric_cols},
    F1_std=("test_F1", "std"), epochs=("epochs", "mean")).reindex(list(MODELS))
summary.to_csv(f"{OUT_RES}/classification_summary.csv")
summary.round(3)

In [ ]:
best_name = summary["val_F1"].idxmax()             # chosen on VALIDATION, not test
best_seed = int(runs[runs.model == best_name].sort_values("val_F1", ascending=False).iloc[0]["seed"])
best_model = trained[(best_name, best_seed)]
p_test = predict_proba(best_model, X_test)
best_metrics = clf_metrics(y_test, p_test)
pred_test = (p_test >= 0.5).astype(int)
print("Best model (by validation F1):", best_name, "seed", best_seed)
print({k: round(v, 3) for k, v in best_metrics.items()})
print(classification_report(y_test, pred_test, target_names=["no diabetes", "diabetes"], digits=3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
s = summary.sort_values("F1")
axes[0].barh(s.index, s["F1"], xerr=s["F1_std"], color="#3b6ea5", capsize=4)
for i, v in enumerate(s["F1"]):
    axes[0].text(v + 0.01, i, f"{v:.3f}", va="center")
axes[0].set_xlim(0, 1); axes[0].set_xlabel("Test F1 (diabetes class), mean ± std over 5 seeds")
axes[0].set_title("Classification model comparison")

cm = confusion_matrix(y_test, pred_test)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1],
            xticklabels=["pred: no", "pred: diabetes"], yticklabels=["true: no", "true: diabetes"])
axes[1].set_title(f"Confusion matrix – {best_name} (test)")

for name in MODELS:
    m = trained[(name, int(runs[runs.model == name].sort_values("val_F1", ascending=False).iloc[0]["seed"]))]
    pr = predict_proba(m, X_test); fpr, tpr, _ = roc_curve(y_test, pr)
    axes[2].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, pr):.3f})")
axes[2].plot([0, 1], [0, 1], "k--", lw=1); axes[2].set_xlabel("False positive rate"); axes[2].set_ylabel("True positive rate")
axes[2].set_title("ROC curves (test)"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/clf_results.png", dpi=150); plt.show()

In [ ]:
h = histories[(best_name, best_seed)]
plt.figure(figsize=(6, 4)); plt.plot(h["train"], label="train"); plt.plot(h["val"], label="validation")
plt.xlabel("epoch"); plt.ylabel("weighted cross-entropy"); plt.title(f"Learning curve – {best_name}"); plt.legend()
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/clf_learning_curve.png", dpi=150); plt.show()

### Decision threshold analysis
The default threshold is 0.5. In screening, missing a diabetic patient (false negative) is usually worse than a false alarm, so we look at how precision/recall/F1 change with the threshold. The threshold is **chosen on the validation set** and then reported on test.

In [ ]:
p_val = predict_proba(best_model, X_val)
thrs = np.round(np.arange(0.2, 0.81, 0.05), 2)
thr_tab = pd.DataFrame([{"threshold": t, **{k: clf_metrics(y_val, p_val, t)[k] for k in ["precision", "recall", "F1"]}}
                        for t in thrs])
best_thr = float(thr_tab.loc[thr_tab.F1.idxmax(), "threshold"])
tuned = clf_metrics(y_test, p_test, best_thr)
print("Validation-optimal threshold:", best_thr)
print("Test @0.5   :", {k: round(best_metrics[k], 3) for k in ["precision", "recall", "F1"]})
print(f"Test @{best_thr}:", {k: round(tuned[k], 3) for k in ["precision", "recall", "F1"]})
plt.figure(figsize=(6, 4))
for k in ["precision", "recall", "F1"]:
    plt.plot(thr_tab.threshold, thr_tab[k], marker="o", label=k)
plt.axvline(best_thr, color="k", ls="--", lw=1); plt.xlabel("threshold on P(diabetes)")
plt.title("Validation precision / recall / F1 vs threshold"); plt.legend()
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/clf_threshold.png", dpi=150); plt.show()

### Permutation importance (best model, test F1 / AUC drop)

In [ ]:
rng = np.random.default_rng(0)
base_auc = roc_auc_score(y_test, p_test)
imp = {}
for j, f in enumerate(FEATURES):
    drops = []
    for _ in range(30):
        Xp = X_test.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
        drops.append(base_auc - roc_auc_score(y_test, predict_proba(best_model, Xp)))
    imp[f] = float(np.mean(drops))
imp = pd.Series(imp).sort_values()
imp.plot.barh(figsize=(7, 4), color="#3b6ea5"); plt.xlabel("Drop in test ROC-AUC when shuffled")
plt.title(f"Permutation importance – {best_name}")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/clf_permutation_importance.png", dpi=150); plt.show()
imp.sort_values(ascending=False)

## 7. Export to ONNX (class approach) and verify

In [ ]:
class ExportWrapper(nn.Module):
    """output1 = softmax probabilities [P(no diabetes), P(diabetes)]."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return torch.softmax(self.m(x), dim=1)

export_model = ExportWrapper(copy.deepcopy(best_model).cpu()).eval().float()
dummy_input = torch.tensor(X_test[:1], dtype=torch.float32)
ONNX_PATH = f"{OUT_WEB}/pima_model.onnx"
export_kw = dict(input_names=["input1"], output_names=["output1"], opset_version=15,
                 do_constant_folding=True,
                 dynamic_axes={"input1": {0: "batch"}, "output1": {0: "batch"}})
try:
    torch.onnx.export(export_model, dummy_input, ONNX_PATH, dynamo=False, **export_kw)
except TypeError:
    torch.onnx.export(export_model, dummy_input, ONNX_PATH, **export_kw)
print("ONNX model saved:", ONNX_PATH, os.path.getsize(ONNX_PATH), "bytes")

In [ ]:
!pip -q install onnxruntime
import onnxruntime as ort
sess = ort.InferenceSession(ONNX_PATH)
onnx_p = sess.run(["output1"], {"input1": X_test.astype(np.float32)})[0][:, 1]
print("max |ONNX - PyTorch| =", float(np.abs(onnx_p - p_test).max()))
print("ONNX test F1 =", f1_score(y_test, (onnx_p >= 0.5).astype(int)))
example = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]], dtype=np.float32)   # row 1, true label = 1
print("Example patient (row 1, true = diabetes) →", sess.run(None, {"input1": example})[0])

In [ ]:
web_meta = {
    "task": "classification", "dataset": "Pima Indians Diabetes",
    "features": FEATURES, "units": UNITS, "zero_as_missing": ZERO_AS_MISSING,
    "defaults": [float(v) for v in medians.numpy()],
    "best_model": best_name, "best_seed": best_seed,
    "threshold_default": 0.5, "threshold_tuned": best_thr,
    "test_metrics": best_metrics, "test_metrics_tuned": tuned,
    "confusion_matrix": cm.tolist(),
    "n_train": int(len(X_train)), "n_val": int(len(X_val)), "n_test": int(len(X_test)),
    "class_counts": {str(k): int(v) for k, v in counts.items()},
    "zeros": {k: int(v) for k, v in zeros.items()},
    "summary": summary.reset_index().to_dict(orient="records"),
    "permutation_importance": imp.sort_values(ascending=False).to_dict(),
    "target_corr": corr[TARGET].drop(TARGET).to_dict(),
    "torch_version": torch.__version__,
}
json.dump(web_meta, open(f"{OUT_WEB}/metrics.json", "w"), indent=2)
json.dump(web_meta, open(f"{OUT_RES}/classification_metrics.json", "w"), indent=2)
print("Saved metrics.json")

## 8. Analysis
Points the report discusses (filled in automatically by `make_report.py`):
* **Imbalance:** accuracy can look good while recall for the diabetic class is poor → F1, recall, MCC, and AUC are the main metrics.
* **Linear vs non-linear:** on this small, noisy dataset logistic regression is usually close to the deep models – extra capacity mostly adds variance (see the ± std).
* **Confusion matrix:** count of missed diabetic patients (false negatives) vs false alarms.
* **Threshold:** lowering the threshold trades precision for recall – appropriate for a screening tool.
* **Correlation matrix / importance:** glucose dominates, followed by BMI, age and pedigree function, consistent with clinical knowledge.

In [ ]:
import shutil
shutil.make_archive("classification_outputs", "zip", "outputs")
try:
    from google.colab import files; files.download("classification_outputs.zip")
except Exception:
    print("Saved classification_outputs.zip")